# Limpieza de datos

In [81]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\ferra\OneDrive\Documentos\Data Projects\ecommerce-sales-analysis\data\raw\online_retail_II - Year 2009-2010.csv", encoding="latin1")

# Columnas del dataframe

In [82]:

print(df.head(20).to_string(index=False))

Invoice StockCode                         Description  Quantity  InvoiceDate  Price  Customer ID        Country
 489434     85048 15CM CHRISTMAS GLASS BALL 20 LIGHTS        12 12/1/09 7:45   6.95      13085.0 United Kingdom
 489434    79323P                  PINK CHERRY LIGHTS        12 12/1/09 7:45   6.75      13085.0 United Kingdom
 489434    79323W                 WHITE CHERRY LIGHTS        12 12/1/09 7:45   6.75      13085.0 United Kingdom
 489434     22041        RECORD FRAME 7" SINGLE SIZE         48 12/1/09 7:45   2.10      13085.0 United Kingdom
 489434     21232      STRAWBERRY CERAMIC TRINKET BOX        24 12/1/09 7:45   1.25      13085.0 United Kingdom
 489434     22064          PINK DOUGHNUT TRINKET POT         24 12/1/09 7:45   1.65      13085.0 United Kingdom
 489434     21871                 SAVE THE PLANET MUG        24 12/1/09 7:45   1.25      13085.0 United Kingdom
 489434     21523  FANCY FONT HOME SWEET HOME DOORMAT        10 12/1/09 7:45   5.95      13085.0 United 

# Valores faltantes por columna

In [83]:
# 1) Faltantes nulos
nulls = df.isna().sum()

# 2) Faltantes tipo texto vacío ("", "   ")
blanks = df.apply(
    lambda col: col.astype(str).str.strip().eq('').sum()
    if col.dtype == 'object' else 0
)

# 3) Resumen
faltantes = pd.DataFrame({
    'nulos': nulls,
    'vacios_texto': blanks,
    'total_faltantes': nulls + blanks
})

faltantes['%_faltantes'] = (faltantes['total_faltantes'] / len(df) * 100).round(2)
faltantes = faltantes.sort_values('total_faltantes', ascending=False)

print(faltantes.to_string())


              nulos  vacios_texto  total_faltantes  %_faltantes
Customer ID  107927             0           107927        20.54
Description    2928             0             2928         0.56
StockCode         0             0                0         0.00
Invoice           0             0                0         0.00
Quantity          0             0                0         0.00
InvoiceDate       0             0                0         0.00
Price             0             0                0         0.00
Country           0             0                0         0.00


In [84]:
def quality_report(df):

    quantity = pd.to_numeric(df['Quantity'], errors='coerce')
    price = pd.to_numeric(df['Price'], errors='coerce')
    date = pd.to_datetime(df['InvoiceDate'], errors='coerce')

    report = {
        'rows': len(df),
        'missing_customer_id': df['Customer ID'].isna().sum(),
        'duplicate_rows': df.duplicated().sum(),
        'bad_invoicedate_format': date.isna().sum(),
        'bad_quantity_format': quantity.isna().sum(),
        'bad_price_format': price.isna().sum(),
        'invalid_quantity_le_0': (quantity <= 0).sum(),
        'invalid_price_le_0': (price <= 0).sum()
    }
    return pd.Series(report)

print(quality_report(df))


C:\Users\ferra\AppData\Local\Temp\ipykernel_16888\4193545347.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  date = pd.to_datetime(df['InvoiceDate'], errors='coerce')


rows                      525461
missing_customer_id       107927
duplicate_rows              6865
bad_invoicedate_format         0
bad_quantity_format            0
bad_price_format               0
invalid_quantity_le_0      12326
invalid_price_le_0          3690
dtype: int64


Identificar formato unico de fecha

In [85]:
df['InvoiceDate'] = pd.to_datetime(
    df['InvoiceDate'],
    format='%m/%d/%y %H:%M',
    errors='coerce'
)
print(df['InvoiceDate'].head())
print(df['InvoiceDate'].isna().sum())  # cuántas no pudo convertir


0   2009-12-01 07:45:00
1   2009-12-01 07:45:00
2   2009-12-01 07:45:00
3   2009-12-01 07:45:00
4   2009-12-01 07:45:00
Name: InvoiceDate, dtype: datetime64[ns]
0


Antes de borrar valores invalidos:
(Quantity < 0 = Devoluciones)
# Filtrar Devoluciones de productos

In [86]:
df_returns = df[pd.to_numeric(df['Quantity'], errors='coerce') < 0].copy()
print (df_returns)

        Invoice StockCode                    Description  Quantity  \
178     C489449     22087       PAPER BUNTING WHITE LACE       -12   
179     C489449    85206A   CREAM FELT EASTER EGG BASKET        -6   
180     C489449     21895  POTTING SHED SOW 'N' GROW SET        -4   
181     C489449     21896             POTTING SHED TWINE        -6   
182     C489449     22083     PAPER CHAIN KIT RETRO SPOT       -12   
...         ...       ...                            ...       ...   
525231   538159     21324                            NaN       -18   
525232   538158     20892                            NaN       -32   
525234   538161    46000S                   Dotcom sales      -100   
525235   538162    46000M                   Dotcom sales      -100   
525282  C538164    35004B    SET OF 3 BLACK FLYING DUCKS        -1   

               InvoiceDate  Price  Customer ID         Country  
178    2009-12-01 10:33:00   2.95      16321.0       Australia  
179    2009-12-01 10:33:00   

In [87]:
df_clean = df[
    (pd.to_numeric(df['Quantity'], errors='coerce') > 0) &
    (pd.to_numeric(df['Price'], errors='coerce') > 0)
].copy()

q = pd.to_numeric(df_clean['Quantity'], errors='coerce')
p = pd.to_numeric(df_clean['Price'], errors='coerce')

print("Quantity <= 0:", (q <= 0).sum())
print("Price <= 0:", (p <= 0).sum())
print("Filas originales:", len(df))
print("Filas limpias:", len(df_clean))


Quantity <= 0: 0
Price <= 0: 0
Filas originales: 525461
Filas limpias: 511566


# Separar en tablas ventas validas de devoluciones

In [88]:
q = pd.to_numeric(df['Quantity'], errors='coerce')
p = pd.to_numeric(df['Price'], errors='coerce')

# 1) Ventas válidas
df_sales = df[(q > 0) & (p > 0)].copy()

# 2) Devoluciones (cantidad negativa)
df_returns = df[(q < 0) & (p > 0)].copy()

# 3) Registros inválidos (para auditar)
df_invalid = df[(q == 0) | (p <= 0)].copy()

print("Original:", len(df))
print("Ventas:", len(df_sales))
print("Devoluciones:", len(df_returns))
print("Inválidos:", len(df_invalid))
print("Suma:", len(df_sales) + len(df_returns) + len(df_invalid))


Original: 525461
Ventas: 511566
Devoluciones: 10205
Inválidos: 3690
Suma: 525461


# Detectar Outliers

In [89]:

# Base para outliers: excluimos solo NaN para poder calcular
work = df.copy()
work['Quantity'] = pd.to_numeric(work['Quantity'], errors='coerce')
work['Price'] = pd.to_numeric(work['Price'], errors='coerce')
work = work.dropna(subset=['Quantity', 'Price'])

def iqr_bounds(s):
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    return low, high

# Límites
q_low, q_high = iqr_bounds(work['Quantity'])
p_low, p_high = iqr_bounds(work['Price'])

# Máscaras de outliers
out_q = (work['Quantity'] < q_low) | (work['Quantity'] > q_high)
out_p = (work['Price'] < p_low) | (work['Price'] > p_high)

# Resumen
print("Quantity -> límite inferior:", round(q_low, 2), "| límite superior:", round(q_high, 2))
print("Quantity outliers:", int(out_q.sum()), f"({out_q.mean()*100:.2f}%)")

print("Price -> límite inferior:", round(p_low, 2), "| límite superior:", round(p_high, 2))
print("Price outliers:", int(out_p.sum()), f"({out_p.mean()*100:.2f}%)")

# Ejemplos de filas atípicas
print("\nEjemplos outliers Quantity:")
print(work.loc[out_q, ['Invoice','StockCode','Quantity','Price','Country']].head(10).to_string(index=False))

print("\nEjemplos outliers Price:")
print(work.loc[out_p, ['Invoice','StockCode','Quantity','Price','Country']].head(10).to_string(index=False))


Quantity -> límite inferior: -12.5 | límite superior: 23.5
Quantity outliers: 57870 (11.01%)
Price -> límite inferior: -3.19 | límite superior: 8.65
Price outliers: 35273 (6.71%)

Ejemplos outliers Quantity:
Invoice StockCode  Quantity  Price        Country
 489434     22041        48   2.10 United Kingdom
 489434     21232        24   1.25 United Kingdom
 489434     22064        24   1.65 United Kingdom
 489434     21871        24   1.25 United Kingdom
 489435     22195        24   1.65 United Kingdom
 489436     22111        24   4.25 United Kingdom
 489438     21329        28   0.98 United Kingdom
 489438     21252        30   1.69 United Kingdom
 489438     21100        30   1.15 United Kingdom
 489438     21033        30   2.00 United Kingdom

Ejemplos outliers Price:
Invoice StockCode  Quantity  Price        Country
 489437     21360         1   9.95 United Kingdom
 489437     35400         2   8.95 United Kingdom
 489439      POST         3  18.00         France
 489444      POS

# Conclusión:
 Se detectaron outliers en Quantity (11.01%) y Price (6.71%) usando IQR.
 Tras revisión de negocio, se consideran valores plausibles del dominio e-commerce
 (compras en volumen/productos de mayor precio), por lo que no se eliminan en esta etapa.


# Transformar Variables para el analisis

In [90]:
df_model = df.copy()

# Tipos correctos
df_model['InvoiceDate'] = pd.to_datetime(df_model['InvoiceDate'], errors='coerce')
df_model['Quantity'] = pd.to_numeric(df_model['Quantity'], errors='coerce')
df_model['Price'] = pd.to_numeric(df_model['Price'], errors='coerce')

# Variables nuevas
df_model['Revenue'] = df_model['Quantity'] * df_model['Price']
df_model['Year'] = df_model['InvoiceDate'].dt.year
df_model['Month'] = df_model['InvoiceDate'].dt.month
df_model['YearMonth'] = df_model['InvoiceDate'].dt.to_period('M').astype(str)
df_model['Day'] = df_model['InvoiceDate'].dt.day
df_model['Weekday'] = df_model['InvoiceDate'].dt.day_name()

# Flag de devolución
df_model['IsReturn'] = df_model['Quantity'] < 0

print(df_model[['InvoiceDate','Quantity','Price','Revenue','YearMonth','Weekday','IsReturn']].head(10).to_string(index=False))


        InvoiceDate  Quantity  Price  Revenue YearMonth Weekday  IsReturn
2009-12-01 07:45:00        12   6.95     83.4   2009-12 Tuesday     False
2009-12-01 07:45:00        12   6.75     81.0   2009-12 Tuesday     False
2009-12-01 07:45:00        12   6.75     81.0   2009-12 Tuesday     False
2009-12-01 07:45:00        48   2.10    100.8   2009-12 Tuesday     False
2009-12-01 07:45:00        24   1.25     30.0   2009-12 Tuesday     False
2009-12-01 07:45:00        24   1.65     39.6   2009-12 Tuesday     False
2009-12-01 07:45:00        24   1.25     30.0   2009-12 Tuesday     False
2009-12-01 07:45:00        10   5.95     59.5   2009-12 Tuesday     False
2009-12-01 07:46:00        12   2.55     30.6   2009-12 Tuesday     False
2009-12-01 07:46:00        12   3.75     45.0   2009-12 Tuesday     False


# Guradar CSVs


In [92]:
from pathlib import Path

output_dir = Path(r"C:\Users\ferra\OneDrive\Documentos\Data Projects\ecommerce-sales-analysis\data\proccesed")
output_dir.mkdir(parents=True, exist_ok=True)

df_sales.to_csv(output_dir / "df_sales.csv", index=False, encoding="utf-8")
df_returns.to_csv(output_dir / "df_return.csv", index=False, encoding="utf-8")
df_invalid.to_csv(output_dir / "df_invalid.csv", index=False, encoding="utf-8")

print("Guardado en:", output_dir)


Guardado en: C:\Users\ferra\OneDrive\Documentos\Data Projects\ecommerce-sales-analysis\data\proccesed


In [93]:
print(df_sales.columns)

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='object')
